# Сравнение accuracy и F1 при дисбалансе классов
**Григорьева Анастасия, № 07**

Гипотеза: при сокращении доли положительного класса модель может сохранить высокую accuracy даже при низкой F1. Проверим, меняется ли выбор лучшей модели в зависимости от метрики.

## Данные и протокол

Один числовой признак: класс 0 генерируется из нормального распределения со средним 0, класс 1 — со средним 2. Стандартное отклонение обоих классов равно 1. Распределения перекрываются, поэтому задача допускает ошибки обоих типов.

На сбалансированной обучающей выборке из 10 000 объектов оценим средние классов. Классификатор ближайшего центра относит объект к классу с ближайшим средним. Базовая модель всегда предсказывает 0. Для проверки независимо от обучения создаём пулы по 10 000 объектов каждого класса. Из них берём первые объекты каждого класса в нужной пропорции, сохраняя размер теста 10 000 и фиксируя модели. Тесты частично пересекаются. Такой дизайн исследует изменение доли классов при оценке уже обученной модели.

In [1]:
from random import Random
from statistics import mean

train_rng = Random(42)
test_rng = Random(43)

def sample(rng, center, size):
    return [rng.gauss(center, 1) for _ in range(size)]

center_0 = mean(sample(train_rng, 0, 5000))
center_1 = mean(sample(train_rng, 2, 5000))
negative_pool = sample(test_rng, 0, 10000)
positive_pool = sample(test_rng, 2, 10000)

def nearest_centroid(value):
    return int(abs(value - center_1) < abs(value - center_0))

print(f'Центры: {center_0:.3f}, {center_1:.3f}')
print(f'Граница решения: {(center_0 + center_1) / 2:.3f}')

Центры: -0.005, 1.981
Граница решения: 0.988


## Метрики

TP — верно найденный класс 1, FP — ложная тревога, FN — пропущенный класс 1, TN — верно найденный класс 0.

- Accuracy = (TP + TN) / N.
- Precision = TP / (TP + FP), доля верных среди предсказанных единиц.
- Recall = TP / (TP + FN), доля найденных среди фактических единиц.
- F1 = 2TP / (2TP + FP + FN), гармоническое среднее precision и recall для класса 1. Если модель не предсказала ни одной единицы, принимаем precision равной нулю.

In [2]:
def evaluate(labels, predictions):
    tn = fp = fn = tp = 0
    for actual, predicted in zip(labels, predictions, strict=True):
        if actual == 0:
            tn += predicted == 0
            fp += predicted == 1
        else:
            fn += predicted == 0
            tp += predicted == 1
    return dict(
        tn=tn, fp=fp, fn=fn, tp=tp,
        accuracy=(tp + tn) / len(labels),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
    )

results = []
for percent in (50, 20, 10, 5, 1):
    positives = percent * 100
    negatives = 10000 - positives
    labels = [0] * negatives + [1] * positives
    test = negative_pool[:negatives] + positive_pool[:positives]
    for name, predictions in (
        ('Всегда 0', [0] * len(test)),
        ('Ближ. центр', [nearest_centroid(x) for x in test]),
    ):
        results.append(dict(percent=percent, model=name, **evaluate(labels, predictions)))

print(f"{'% кл. 1':>7}  {'Модель':<12} {'Acc':>6} {'F1':>6} {'Prec':>6} {'Recall':>6}")
for row in results:
    print(f"{row['percent']:>7}  {row['model']:<12} "
          f"{row['accuracy']:6.3f} {row['f1']:6.3f} "
          f"{row['precision']:6.3f} {row['recall']:6.3f}")

% кл. 1  Модель          Acc     F1   Prec Recall
     50  Всегда 0      0.500  0.000  0.000  0.000
     50  Ближ. центр   0.844  0.845  0.841  0.848
     20  Всегда 0      0.800  0.000  0.000  0.000
     20  Ближ. центр   0.842  0.680  0.571  0.841
     10  Всегда 0      0.900  0.000  0.000  0.000
     10  Ближ. центр   0.840  0.509  0.367  0.830
      5  Всегда 0      0.950  0.000  0.000  0.000
      5  Ближ. центр   0.841  0.345  0.217  0.838
      1  Всегда 0      0.990  0.000  0.000  0.000
      1  Ближ. центр   0.842  0.093  0.049  0.810


In [3]:
for row in results[-2:]:
    print(row['model'])
    print('             Прогноз 0   Прогноз 1')
    print(f"Факт 0       {row['tn']:9d}   {row['fp']:9d}")
    print(f"Факт 1       {row['fn']:9d}   {row['tp']:9d}")

Всегда 0
             Прогноз 0   Прогноз 1
Факт 0            9900           0
Факт 1             100           0
Ближ. центр
             Прогноз 0   Прогноз 1
Факт 0            8335        1565
Факт 1              19          81


## Выводы

При 1% положительных примеров базовая модель получает accuracy 0.990 и F1 0.000. Она пропускает все 100 единиц. Классификатор ближайшего центра находит 81 из 100, но даёт 1565 ложных тревог. Его accuracy равна 0.842, а F1 — 0.093. При 1% accuracy выбирает базовую модель, а F1 класса 1 — классификатор ближайшего центра.

У ближайшего центра recall при 1% составляет 0.810, а precision — 0.049. Даже при высоком recall многочисленный класс 0 даёт много ложных тревог. Из-за этого падают precision и F1. Для задачи, в которой важно находить редкие события, полезно показывать F1 вместе с precision, recall и числом ошибок каждого типа. Если ошибки имеют разные последствия, решение также зависит от их стоимости.

Ограничения: синтетические данные, один признак, один запуск и фиксированная граница решения. Эксперимент меняет баланс только на тесте. Он не измеряет эффект дисбаланса при обучении или статистическую неопределённость оценок.